# Análisis exploratorio de datos de anuncios de autos

## Objetivo

El objetivo de este análisis es explorar y preparar el conjunto de datos de anuncios de vehículos usados para identificar patrones, valores faltantes, valores atípicos y características relevantes de los vehículos.

Durante el análisis se hara la limpieza de los datos y se utilizarán visualizaciones para comprender mejor la distribución de las principales variables.

In [53]:
import pandas as pd
import plotly.express as px

## 1. Carga y exploración inicial de los datos

Primero se carga el conjunto de datos y se revisan sus dimensiones, columnas, tipos de datos y valores faltantes.

In [54]:
df = pd.read_csv('../vehicles_us.csv')

df.head()

,price,model_year,model,condition,cylinders,fuel,odometer,transmission,type,paint_color,is_4wd,date_posted,days_listed
0,9400,2011.0,bmw x5,good,6.0,gas,145000.0,automatic,SUV,NaN,1.0,2018-06-23,19
1,25500,NaN,ford f-150,good,6.0,gas,88705.0,automatic,pickup,white,1.0,2018-10-19,50
2,5500,2013.0,hyundai sonata,like new,4.0,gas,110000.0,automatic,sedan,red,NaN,2019-02-07,79
3,1500,2003.0,ford f-150,fair,8.0,gas,NaN,automatic,pickup,NaN,NaN,2019-03-22,9
4,14900,2017.0,chrysler 200,excellent,4.0,gas,80903.0,automatic,sedan,black,NaN,2019-04-02,28


In [55]:
df.shape

(51525, 13)

In [56]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 51525 entries, 0 to 51524
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   price         51525 non-null  int64  
 1   model_year    47906 non-null  float64
 2   model         51525 non-null  str    
 3   condition     51525 non-null  str    
 4   cylinders     46265 non-null  float64
 5   fuel          51525 non-null  str    
 6   odometer      43633 non-null  float64
 7   transmission  51525 non-null  str    
 8   type          51525 non-null  str    
 9   paint_color   42258 non-null  str    
 10  is_4wd        25572 non-null  float64
 11  date_posted   51525 non-null  str    
 12  days_listed   51525 non-null  int64  
dtypes: float64(4), int64(2), str(7)
memory usage: 7.7 MB


## 2. Limpieza y preparación de los datos

Se identificaron valores faltantes en las columnas `model_year`, `cylinders`, `odometer`, `paint_color` e `is_4wd`.

Los valores faltantes se tratarán utilizando métodos adecuados según el significado de cada variable.

In [57]:
df.isna().sum()

price               0
model_year       3619
model               0
condition           0
cylinders        5260
fuel                0
odometer         7892
transmission        0
type                0
paint_color      9267
is_4wd          25953
date_posted         0
days_listed         0
dtype: int64

In [58]:
(df.isna().sum() / len(df)) * 100

price            0.000000
model_year       7.023775
model            0.000000
condition        0.000000
cylinders       10.208637
fuel             0.000000
odometer        15.316836
transmission     0.000000
type             0.000000
paint_color     17.985444
is_4wd          50.369723
date_posted      0.000000
days_listed      0.000000
dtype: float64

In [59]:
df['is_4wd'].value_counts(dropna=False)

is_4wd
NaN    25953
1.0    25572
Name: count, dtype: int64

In [60]:
df['is_4wd'] = df['is_4wd'].fillna(0)

df['is_4wd'].value_counts(dropna=False)

is_4wd
0.0    25953
1.0    25572
Name: count, dtype: int64

In [61]:
df[['model', 'model_year']].head(20)

,model,model_year
0,bmw x5,2011.0
1,ford f-150,NaN
2,hyundai sonata,2013.0
3,ford f-150,2003.0
4,chrysler 200,2017.0
5,chrysler 300,2014.0
6,toyota camry,2015.0
7,honda pilot,2013.0
8,kia sorento,2012.0
9,honda pilot,2008.0


In [62]:
df.groupby('model')['model_year'].median().head(20)

model
acura tl                         2007.0
bmw x5                           2010.0
buick enclave                    2012.0
cadillac escalade                2009.0
chevrolet camaro                 2013.0
chevrolet camaro lt coupe 2d     2017.0
chevrolet colorado               2015.0
chevrolet corvette               2000.0
chevrolet cruze                  2014.0
chevrolet equinox                2013.0
chevrolet impala                 2010.0
chevrolet malibu                 2012.0
chevrolet silverado              2008.0
chevrolet silverado 1500         2011.0
chevrolet silverado 1500 crew    2016.0
chevrolet silverado 2500hd       2010.0
chevrolet silverado 3500hd       2013.0
chevrolet suburban               2008.0
chevrolet tahoe                  2009.0
chevrolet trailblazer            2005.0
Name: model_year, dtype: float64

In [63]:
df['model_year'] = df['model_year'].fillna(
    df.groupby('model')['model_year'].transform('median')
)

df['model_year'].isna().sum()

np.int64(0)

In [64]:
df['model_year'].dtype

dtype('float64')

In [65]:
df['model_year'] = df['model_year'].astype(int)

df['model_year'].dtype

df[['model', 'model_year']].head(10)

,model,model_year
0,bmw x5,2011
1,ford f-150,2011
2,hyundai sonata,2013
3,ford f-150,2003
4,chrysler 200,2017
5,chrysler 300,2014
6,toyota camry,2015
7,honda pilot,2013
8,kia sorento,2012
9,honda pilot,2008


In [66]:
df['cylinders'].value_counts(dropna=False).sort_index()

cylinders
3.0        34
4.0     13864
5.0       272
6.0     15700
8.0     15844
10.0      549
12.0        2
NaN      5260
Name: count, dtype: int64

In [67]:
df.groupby('model')['cylinders'].median().head(20)

model
acura tl                         6.0
bmw x5                           6.0
buick enclave                    6.0
cadillac escalade                8.0
chevrolet camaro                 6.0
chevrolet camaro lt coupe 2d     6.0
chevrolet colorado               5.0
chevrolet corvette               8.0
chevrolet cruze                  4.0
chevrolet equinox                4.0
chevrolet impala                 6.0
chevrolet malibu                 4.0
chevrolet silverado              8.0
chevrolet silverado 1500         8.0
chevrolet silverado 1500 crew    8.0
chevrolet silverado 2500hd       8.0
chevrolet silverado 3500hd       8.0
chevrolet suburban               8.0
chevrolet tahoe                  8.0
chevrolet trailblazer            6.0
Name: cylinders, dtype: float64

In [68]:
df['cylinders'] = df['cylinders'].fillna(
    df.groupby('model')['cylinders'].transform('median')
)

df['cylinders'].isna().sum()

np.int64(0)

In [69]:
df['cylinders'] = df['cylinders'].astype(int)

df['cylinders'].dtype

dtype('int64')

In [70]:
df['odometer'].describe()

count     43633.000000
mean     115553.461738
std       65094.611341
min           0.000000
25%       70000.000000
50%      113000.000000
75%      155000.000000
max      990000.000000
Name: odometer, dtype: float64

In [71]:
df.groupby('model_year')['odometer'].median().head(20)

model_year
1908    169328.0
1929         NaN
1936     30000.0
1948      4000.0
1949      1800.0
1954      3565.0
1955     47180.0
1958     32991.5
1960     16000.0
1961     66000.0
1962     72000.0
1963     40487.0
1964     58000.0
1965     51330.5
1966     63070.0
1967     94000.0
1968     31362.0
1969     42509.5
1970     83890.0
1971     56329.0
Name: odometer, dtype: float64

In [72]:
df.groupby('model_year')['odometer'].median().tail(20)

model_year
2000    175000.0
2001    179183.0
2002    160000.0
2003    161397.0
2004    156640.0
2005    153108.0
2006    150920.0
2007    142000.0
2008    140000.0
2009    131565.0
2010    127168.5
2011    123025.0
2012    110000.0
2013     99840.0
2014     90000.0
2015     78285.5
2016     53998.5
2017     41000.0
2018     20674.0
2019     14151.5
Name: odometer, dtype: float64

In [73]:
df['odometer'] = df['odometer'].fillna(
    df.groupby('model_year')['odometer'].transform('median')
)

df['odometer'].isna().sum()

np.int64(1)

In [74]:
df[df['odometer'].isna()]

,price,model_year,model,condition,cylinders,fuel,odometer,transmission,type,paint_color,is_4wd,date_posted,days_listed
45694,18000,1929,ford f-150,good,8,gas,NaN,manual,other,silver,0.0,2018-11-18,59


In [75]:
df['odometer'] = df['odometer'].fillna(
    df['odometer'].median()
)

df['odometer'].isna().sum()

np.int64(0)

In [76]:
df['paint_color'].value_counts(dropna=False)

paint_color
white     10029
NaN        9267
black      7692
silver     6244
grey       5037
blue       4475
red        4421
green      1396
brown      1223
custom     1153
yellow      255
orange      231
purple      102
Name: count, dtype: int64

In [77]:
df['paint_color'] = df['paint_color'].fillna('unknown')

df['paint_color'].value_counts(dropna=False)

paint_color
white      10029
unknown     9267
black       7692
silver      6244
grey        5037
blue        4475
red         4421
green       1396
brown       1223
custom      1153
yellow       255
orange       231
purple       102
Name: count, dtype: int64

In [78]:
df.isna().sum()

price           0
model_year      0
model           0
condition       0
cylinders       0
fuel            0
odometer        0
transmission    0
type            0
paint_color     0
is_4wd          0
date_posted     0
days_listed     0
dtype: int64

## 3. Corrección de tipos de datos

Algunas columnas tenían tipos de datos que no representaban correctamente su contenido. Se realizaron conversiones para facilitar el análisis posterior.

In [79]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 51525 entries, 0 to 51524
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   price         51525 non-null  int64  
 1   model_year    51525 non-null  int64  
 2   model         51525 non-null  str    
 3   condition     51525 non-null  str    
 4   cylinders     51525 non-null  int64  
 5   fuel          51525 non-null  str    
 6   odometer      51525 non-null  float64
 7   transmission  51525 non-null  str    
 8   type          51525 non-null  str    
 9   paint_color   51525 non-null  str    
 10  is_4wd        51525 non-null  float64
 11  date_posted   51525 non-null  str    
 12  days_listed   51525 non-null  int64  
dtypes: float64(2), int64(4), str(7)
memory usage: 7.7 MB


In [80]:
df['odometer'] = df['odometer'].astype(int)

df['odometer'].dtype

dtype('int64')

In [81]:
df['is_4wd'] = df['is_4wd'].astype(int)

df['is_4wd'].value_counts()

is_4wd
0    25953
1    25572
Name: count, dtype: int64

In [82]:
df['date_posted'] = pd.to_datetime(df['date_posted'])

df['date_posted'].dtype

dtype('<M8[us]')

In [83]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 51525 entries, 0 to 51524
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   price         51525 non-null  int64         
 1   model_year    51525 non-null  int64         
 2   model         51525 non-null  str           
 3   condition     51525 non-null  str           
 4   cylinders     51525 non-null  int64         
 5   fuel          51525 non-null  str           
 6   odometer      51525 non-null  int64         
 7   transmission  51525 non-null  str           
 8   type          51525 non-null  str           
 9   paint_color   51525 non-null  str           
 10  is_4wd        51525 non-null  int64         
 11  date_posted   51525 non-null  datetime64[us]
 12  days_listed   51525 non-null  int64         
dtypes: datetime64[us](1), int64(6), str(6)
memory usage: 7.2 MB


## 4. Comprobación de registros duplicados

Se verificó la existencia de filas completamente duplicadas en el conjunto de datos.

In [84]:
df.duplicated().sum()

np.int64(0)

No se encontraron registros completamente duplicados en el conjunto de datos.

In [85]:
df[['price', 'model_year', 'cylinders', 'odometer', 'days_listed']].describe()

,price,model_year,cylinders,odometer,days_listed
count,51525.000000,51525.000000,51525.000000,51525.000000,51525.00000
mean,12132.464920,2009.793557,6.121067,115199.271771,39.55476
std,10040.803015,6.099381,1.657457,62082.383589,28.20427
min,1.000000,1908.000000,3.000000,0.000000,0.00000
25%,5000.000000,2007.000000,4.000000,73500.000000,19.00000
50%,9000.000000,2011.000000,6.000000,114074.000000,33.00000
75%,16839.000000,2014.000000,8.000000,152827.000000,53.00000
max,375000.000000,2019.000000,12.000000,990000.000000,271.00000


## 5. Análisis de la variable `price`

Se analizó la distribución de los precios para identificar valores atípicos que pudieran afectar las estadísticas y visualizaciones.

Se detectó una cantidad considerable de anuncios con un precio de $1, valor que no parece representar un precio de venta real.

In [86]:
df['price'].sort_values().head(20)

12186    1
40638    1
14644    1
49385    1
11298    1
11297    1
11296    1
11295    1
16868    1
16872    1
29030    1
11320    1
11319    1
11318    1
11317    1
14649    1
41228    1
14647    1
14646    1
14645    1
Name: price, dtype: int64

In [87]:
df[df['price'] == 1].head(20)

,price,model_year,model,condition,cylinders,fuel,odometer,transmission,type,paint_color,is_4wd,date_posted,days_listed
405,1,2014,chevrolet camaro,excellent,6,gas,71310,automatic,coupe,unknown,0,2018-07-14,29
3063,1,1998,chevrolet silverado,good,8,gas,164000,automatic,pickup,unknown,1,2018-10-11,49
3808,1,2007,chevrolet tahoe,good,8,gas,200,automatic,SUV,red,0,2019-03-18,63
3902,1,1996,ford f-150,fair,8,gas,163000,manual,truck,white,0,2019-02-23,54
4140,1,2004,chevrolet silverado,excellent,8,diesel,83000,automatic,pickup,unknown,1,2019-02-04,14
5612,1,2006,gmc sierra,excellent,8,gas,192960,automatic,truck,white,0,2018-10-29,39
5700,1,2015,ram 2500,excellent,6,diesel,103549,automatic,truck,red,1,2018-07-31,45
5718,1,2010,toyota tacoma,good,6,gas,168955,automatic,truck,silver,1,2018-12-31,40
5907,1,2011,toyota tacoma,good,6,gas,168955,automatic,truck,unknown,1,2019-01-03,12
6012,1,2015,ram 2500,excellent,6,diesel,103549,automatic,truck,red,1,2019-02-17,26


In [88]:
(df['price'] == 1).sum()

np.int64(798)

In [89]:
df['price'].value_counts().sort_index().head(30)

price
1      798
3        1
5        1
6        1
9        1
10       1
11       1
12       3
15       3
20       1
24       1
25       2
28       1
32       1
35       2
36       1
39       3
65       1
69      34
80       2
85       4
105      1
111      1
147      1
155      3
169      1
171      1
176      3
179      1
180      2
Name: count, dtype: int64

In [90]:
df = df[df['price'] != 1]

df.shape

(50727, 13)

In [91]:
(df['price'] == 1).sum()

np.int64(0)

Se identificaron 798 anuncios con un precio de $1. Debido a la frecuencia de este valor y a que no representa un precio razonable de venta para los vehículos observados, estos registros fueron excluidos del análisis.

In [92]:
df.sort_values('price', ascending=False).head(20)

,price,model_year,model,condition,cylinders,fuel,odometer,transmission,type,paint_color,is_4wd,date_posted,days_listed
12504,375000,1999,nissan frontier,good,6,gas,115000,automatic,pickup,blue,1,2018-05-19,21
11359,300000,2015,ram 2500,excellent,6,diesel,78285,automatic,truck,grey,1,2018-10-15,39
27375,189000,2014,ford f-150,good,6,gas,151248,automatic,truck,black,0,2018-09-25,72
34389,189000,2014,ford f-150,good,6,gas,151248,automatic,truck,black,0,2019-02-02,28
1668,189000,2014,ford f-150,good,6,gas,151248,automatic,truck,unknown,0,2019-03-20,21
33434,189000,2014,ford f-150,good,6,gas,151248,automatic,truck,black,0,2019-02-05,102
1309,189000,2014,ford f-150,good,6,gas,151248,automatic,truck,black,0,2019-03-02,56
30634,189000,2014,ford f-150,good,6,gas,90000,automatic,truck,black,0,2018-07-21,42
34206,175000,2004,gmc sierra 2500hd,good,8,diesel,149000,automatic,truck,grey,1,2018-08-25,57
41748,145000,2008,toyota tundra,like new,8,gas,140000,automatic,truck,red,1,2018-06-16,29


### Valores atípicos en los precios

Se analizaron los percentiles de la variable `price` para conocer la distribución de los precios y detectar valores extremadamente altos.

In [93]:
df['price'].quantile([0.90, 0.95, 0.99, 0.995, 0.999])

0.900    25500.000
0.950    30500.000
0.990    43999.000
0.995    49900.000
0.999    64590.818
Name: price, dtype: float64

El 99.9% de los anuncios tiene un precio aproximado de $64,591 o menor. Sin embargo, existen algunos anuncios con precios considerablemente superiores.

Estos registros se conservarán en el conjunto de datos, pero se tendrá en cuenta su efecto al realizar las visualizaciones.

In [94]:
print("Más de $50,000:", (df['price'] > 50000).sum())
print("Más de $65,000:", (df['price'] > 65000).sum())
print("Más de $75,000:", (df['price'] > 75000).sum())
print("Más de $100,000:", (df['price'] > 100000).sum())

Más de $50,000: 227
Más de $65,000: 43
Más de $75,000: 26
Más de $100,000: 17


In [95]:
df[df['price'] > 100000].sort_values('price', ascending=False)

,price,model_year,model,condition,cylinders,fuel,odometer,transmission,type,paint_color,is_4wd,date_posted,days_listed
12504,375000,1999,nissan frontier,good,6,gas,115000,automatic,pickup,blue,1,2018-05-19,21
11359,300000,2015,ram 2500,excellent,6,diesel,78285,automatic,truck,grey,1,2018-10-15,39
1309,189000,2014,ford f-150,good,6,gas,151248,automatic,truck,black,0,2019-03-02,56
1668,189000,2014,ford f-150,good,6,gas,151248,automatic,truck,unknown,0,2019-03-20,21
27375,189000,2014,ford f-150,good,6,gas,151248,automatic,truck,black,0,2018-09-25,72
33434,189000,2014,ford f-150,good,6,gas,151248,automatic,truck,black,0,2019-02-05,102
34389,189000,2014,ford f-150,good,6,gas,151248,automatic,truck,black,0,2019-02-02,28
30634,189000,2014,ford f-150,good,6,gas,90000,automatic,truck,black,0,2018-07-21,42
34206,175000,2004,gmc sierra 2500hd,good,8,diesel,149000,automatic,truck,grey,1,2018-08-25,57
41748,145000,2008,toyota tundra,like new,8,gas,140000,automatic,truck,red,1,2018-06-16,29


## 6. Análisis exploratorio mediante visualizaciones

### Distribución de precios

Primero se observa la distribución completa de los precios.

In [96]:
fig = px.histogram(
    df,
    x='price',
    title='Distribución de precios de vehículos'
)

fig.show()

La distribución presenta una fuerte asimetría hacia la derecha. Algunos precios extremadamente altos amplían considerablemente el eje horizontal y dificultan observar la distribución de la mayoría de los vehículos.

In [97]:
fig = px.histogram(
    df[df['price'] <= 65000],
    x='price',
    title='Distribución de precios de vehículos hasta $65,000'
)

fig.show()

Al limitar la visualización a vehículos con precios de hasta $65,000 se observa con mayor claridad la distribución principal.

La mayor concentración de anuncios se encuentra en los rangos de precios bajos y medios, mientras que la frecuencia disminuye progresivamente conforme aumenta el precio. La distribución presenta una cola hacia la derecha.

In [ ]:
aqui sigo 